# 08 — Tracing & Observability

This notebook covers core requirement 3, verbatim from the official spec:

> **Tracing and observability.** You must instrument your system with a tracing platform such as LangSmith, Langfuse, or Arize Phoenix. By the end of the project, you should be able to open a trace of a real multi-step run and explain exactly what your agent did, in what order, and why.

**Why LangSmith**: `factory_floor/agent.py`'s Diagnostic Agent is built on `langchain.agents.create_agent` (LangGraph under the hood), which wires up to LangSmith with only environment variables and no extra instrumentation code — this was one of the reasons LangGraph was chosen over a manual tool-calling loop back in the Orchestrator Agent milestone (see `CLAUDE.md`'s 2026-08-19 note).

**A real environment wrinkle, handled rather than ignored**: the `.env` this project shares with unrelated bootcamp labs already has `LANGSMITH_TRACING=true` pinned to a project called `lca-lc-foundation`. Rather than edit `.env` (the owner asked for it to be left alone — it's reused across several unrelated projects), `factory_floor/tracing.py::configure_tracing()` overrides `LANGSMITH_PROJECT` to a dedicated `"factory-floor"` project at import time (`factory_floor/__init__.py` runs this on any `from factory_floor.X import Y`), so this project's traces are isolated from the others sharing that key. This notebook is the entry point that proves it actually works.

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import factory_floor  # noqa: F401 -- import alone triggers configure_tracing()
from factory_floor.tracing import is_tracing_enabled, tracing_endpoint, run_url
from factory_floor.config import COLLECTION_NAME, VECTOR_DIR
from factory_floor.vectorstore import get_embeddings, load_vectorstore
from factory_floor.rag import ask, build_retriever, get_llm
from factory_floor.vision import load_classifier, classify_defect_trained
from factory_floor.defect_dataset import load_manifest
from factory_floor.agent import run_diagnostic_agent
from langchain_core.tracers.langchain import wait_for_all_tracers

embeddings = get_embeddings()
vectorstore = load_vectorstore(VECTOR_DIR, COLLECTION_NAME, embeddings=embeddings)
llm = get_llm()

print('LangSmith project:', os.environ['LANGSMITH_PROJECT'])
print('LangSmith endpoint:', tracing_endpoint())
print('Tracing enabled:', is_tracing_enabled())
print('API key present:', bool(os.environ.get('LANGSMITH_API_KEY')))  # never print the key itself -- this notebook's outputs get committed

assert os.environ['LANGSMITH_PROJECT'] == 'factory-floor', 'Expected the code-level override, not the shared .env value'
assert is_tracing_enabled(), 'LANGSMITH_TRACING must be true for this notebook to produce real traces'

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LangSmith project: factory-floor
LangSmith endpoint: https://eu.api.smith.langchain.com
Tracing enabled: True
API key present: True


## What gets attached to every run, and why

Every `run_diagnostic_agent()` call is tagged and named via `factory_floor/tracing.py::trace_config()` before it reaches `agent.invoke()`:
- `run_name="diagnostic_agent"` — so the trace list is legible at a glance instead of showing generic `AgentExecutor` entries.
- `tags=["factory_floor", "diagnostic_agent", "machine:<id>"]` — filterable in the LangSmith UI, e.g. to pull up every run against one specific machine.
- `metadata` — `machine_id`, `language`, `llm_model`, whether a photo was involved, how many prior conversation turns were carried in, and the exact list of tools that were *available* for that run (not just the ones called) — this last one is what lets you open a `GENERAL` question's trace and confirm the history tool was never even offered, not just unused.

`rag.py::ask()` (the baseline pipeline used for comparison in the evaluation notebook) is tagged similarly but is not a single Runnable — see the "Contrast with the baseline pipeline" section below for why that needed a different tracing mechanism (`@traceable`, not `config=`).

In [2]:
from factory_floor.tracing import trace_config

example = trace_config(
    'diagnostic_agent',
    tags=['diagnostic_agent', 'machine:VFD-06'],
    metadata={'machine_id': 'VFD-06', 'language': 'English', 'llm_model': 'gpt-4.1-mini'},
)
print(example)

{'run_name': 'diagnostic_agent', 'tags': ['factory_floor', 'diagnostic_agent', 'machine:VFD-06'], 'metadata': {'component': 'diagnostic_agent', 'machine_id': 'VFD-06', 'language': 'English', 'llm_model': 'gpt-4.1-mini'}, 'run_id': UUID('0724c09b-248f-4da8-99c9-ffd302c388fe')}


## The multi-step scenario

A genuinely multi-step run needs to actually decide between tools, not just call one on autopilot. `VFD-06`'s simulated maintenance history already contains an `F30059` (internal fan fault) event, so a question that asks both "what should I check" *and* "has this happened before" gives the agent a real reason to call **both** `get_maintenance_history` and `search_manuals`, in whichever order it judges sensible — not because the notebook forces it to, but because the question genuinely calls for both. This was verified live before writing it into this notebook (the agent consistently checks history first, then the manual) rather than assumed.

In [3]:
vfd_retriever = build_retriever(vectorstore, k=5, equipment_type='VFD')

multi_step_question = (
    'This VFD tripped again with F30059, internal fan fault. Has this specific machine '
    'had this problem before, and what does the manual say should be checked?'
)

agent_result = run_diagnostic_agent(
    multi_step_question, vfd_retriever, machine_id='VFD-06', llm=llm, language='English',
)
wait_for_all_tracers()
agent_trace_url = run_url(agent_result['run_id'])

print('ANSWER:\n', agent_result['answer'])
print('\nTOOLS USED, IN ORDER:', [t['tool'] for t in agent_result['tool_trace']])
print('\nrun_id:', agent_result['run_id'])
print('trace URL:', agent_trace_url)

assert len(agent_result['tool_trace']) >= 2, 'Expected the agent to consult more than one tool for a real multi-step run'

ANSWER:
 The specific machine VFD-06 has had the F30059 internal fan fault before, with a recorded fault on 2022-06-25 where the internal power unit cooling fan was replaced.

According to the Siemens SINAMICS G120C manuals (Siemens_SINAMICS_G120C_List_Manual.pdf, page 737 and Siemens_G120_CU240BE2_List_Manual.pdf, page 915), the recommended action for fault F30059 "internal fan defective" is:

Safety precautions:
- Isolate and de-energize the drive.
- Apply lockout/tagout.
- Wait for the DC link capacitors to discharge before touching any terminal.
- Verify absence of voltage.
- Work should be performed by qualified personnel only.

Then:
- Check the internal fan.
- Replace the fan if necessary.
- Reset the operating hours counter parameters p0251 and p0254 after replacement.

Please follow these steps carefully to address the internal fan fault. If the fan replacement was already done recently and the fault recurs, further investigation into the fan power supply or control circuit ma

## Reading the trace

The cell below reconstructs the same story `_extract_tool_trace()` already gives the Streamlit UI's "Tools used" line — printed here so this notebook stands on its own even for a reader without LangSmith access. Opening the real trace at the URL above gives the same sequence plus what this local view can't: exact latency per step, token usage, and the literal prompt text sent to the LLM at each turn.

In [4]:
print(f"Question: {multi_step_question}\n")
for i, step in enumerate(agent_result['tool_trace'], 1):
    print(f"{i}. agent called {step['tool']}({step['input']})")
    print(f"   -> {step['output_preview'][:200]}...\n")
print(f"{len(agent_result['tool_trace']) + 1}. agent produced the final answer shown above.")
print('\nThis local reconstruction and the LangSmith trace tree tell the same story --')
print('the tree just also shows timing, token counts, and the exact prompt sent at each step.')

Question: This VFD tripped again with F30059, internal fan fault. Has this specific machine had this problem before, and what does the manual say should be checked?

1. agent called search_manuals({'query': 'F30059 internal fan fault'})
   -> [SOURCE 1] Siemens_SINAMICS_G120C_List_Manual.pdf, page 737
Remedy For the fan involved, carry out the following:
- replace the fan.
- reset the operating hours counter (p0251, p0254).
See also: p0251...

2. agent called get_maintenance_history({})
   -> - 2022-06-25 [fault] F30059 — action taken: Replaced the internal power unit cooling fan
- 2023-11-26 [repair] F30002 — action taken: Checked braking resistor and ramp-down time; verified line supply ...

3. agent produced the final answer shown above.

This local reconstruction and the LangSmith trace tree tell the same story --
the tree just also shows timing, token counts, and the exact prompt sent at each step.


## Contrast with the baseline pipeline

`rag.ask()` is plain Python (`contextualize_question` → `retriever.invoke()` → one `llm.invoke()`), not a single LangChain Runnable, so passing `config=` to sub-calls would have produced several unrelated sibling root traces instead of one comparable chain. It's wrapped in `@traceable` instead (see `rag.py::_ask_traced`), which groups it into one `rag_baseline` root run per call — directly comparable to the agent's one root run, just structurally simpler (no tool children, because there are no tools).

A genuinely useful trace-reading exercise for the presentation: open the agent's run above and the baseline's run below side by side. The agent's tree shows a real decision (which tool, in what order); the baseline's is always the same two-LLM-call shape regardless of the question, because it has no way to decide anything.

In [5]:
baseline_result = ask(multi_step_question, vfd_retriever, llm=llm, language='English')
wait_for_all_tracers()
baseline_trace_url = run_url(baseline_result['run_id'])

print('ANSWER:\n', baseline_result['answer'])
print('\nrun_id:', baseline_result['run_id'])
print('trace URL:', baseline_trace_url)
print("\nNote: ask() has no tools and therefore no tool_trace at all -- its LangSmith run is a single")
print("'rag_baseline' chain with two LLM calls as children (contextualize + answer), never a multi-step")
print("agent->tool->agent tree. That contrast is itself part of what the trace should make legible.")

ANSWER:
 The VFD has tripped with fault F30059, indicating an internal fan fault. According to the documentation, this specific fault has been recorded before on this machine (implied by your question, but no explicit history is given in the retrieved context).

The manual advises the following checks and remedies for fault F30059:
- Check the internal fan and replace it if necessary [SOURCE 1, p. 737; SOURCE 2, p. 399].
- Reset the operating hours counter for the fan (parameters p0251, p0254) after replacement [SOURCE 1, p. 737].
- Verify that the fan is running properly and that fan filter elements are clean [SOURCE 2, p. 399].
- Check that the ambient temperature is within the permissible range to avoid overheating [SOURCE 2, p. 399; SOURCE 3, p. 735].
- If the fault persists, consider that insufficient cooling or fan failure is the cause [SOURCE 3, p. 735].

In summary, verify the internal fan operation, replace the fan if faulty, clean or check fan filters, ensure ambient temperat

## Bonus trace: agentic judgment without any tool call at all

The Orchestrator Agent milestone's central test (`notebooks/07_orchestrator_agent.ipynb`, Scenario 2) is worth tracing too, for a different reason: it's a real multi-step *reasoning* run that calls **zero** tools. A held-out photo classified as `structural_damage` is paired with a text description that downplays it as "just a cosmetic scuff" — the agent has to notice the conflict and ask a clarifying question instead of guessing. Its trace won't show a tool-call tree, but it will show the full reasoning turn, which is exactly the "why" the spec asks for — a case where the interesting decision is *not* calling a tool.

In [6]:
manifest_rows = load_manifest()
damage_row = next(r for r in manifest_rows if r['split'] == 'test' and r['coarse_label'] == 'structural_damage')
image_path = PROJECT_ROOT / damage_row['filepath']

clf, label_list = load_classifier()
classification = classify_defect_trained(image_path, clf, label_list)
vision_context = (
    f"Vision analysis of the uploaded photo: predicted condition = {classification['predicted_label']} "
    f"(confidence {classification['confidence']:.0%}, defective)."
)

conflict_result = run_diagnostic_agent(
    'Just a tiny cosmetic scuff, nothing serious, probably fine to keep running.',
    vfd_retriever, machine_id='VFD-06', vision_context=vision_context, llm=llm, language='English',
)
wait_for_all_tracers()
conflict_trace_url = run_url(conflict_result['run_id'])

print('Vision classification:', classification['predicted_label'], f"({classification['confidence']:.0%} confidence)")
print('\nANSWER:\n', conflict_result['answer'])
print('\ntrace URL:', conflict_trace_url)

assert '?' in conflict_result['answer'], 'Expected the agent to ask a clarifying question here'
print('\nConfirmed: the agent asked instead of guessing -- the trace records exactly why (conflicting signals), not just that it happened.')

Vision classification: structural_damage (82% confidence)

ANSWER:
 The operator describes only a minor cosmetic scuff on the VFD, but the vision analysis indicates structural damage with high confidence. This discrepancy suggests the damage may be more serious than initially assessed visually.

Safety precautions:
- Before any inspection or intervention, isolate and de-energize the VFD.
- Apply lockout/tagout procedures to ensure the drive cannot be accidentally powered.
- Wait for the DC link capacitors to discharge fully before touching any terminals.
- Verify absence of voltage with appropriate instruments.
- Only qualified personnel should perform physical inspection or repairs.

Next steps:
1. Perform a detailed physical inspection of the VFD enclosure and components to confirm the extent of the structural damage.
2. Check for any cracks, broken parts, or compromised seals that could affect the VFD's operation or safety.
3. Review the VFD's fault history to see if any faults or a

## Deliverable: what to screenshot

The spec's deliverable is screenshots or shared links demonstrating the observability of a real multi-step run — this last step is inherently manual (it needs your own LangSmith login), so here's exactly what to capture from the URLs printed above, in the `factory-floor` project:

1. **The multi-step agent trace** (URL printed in the "multi-step scenario" cell) — expand the tree so `diagnostic_agent → get_maintenance_history → search_manuals → diagnostic_agent` (or whichever order came out) is fully visible, showing the agent chose two different tools in sequence.
2. **One tool's child run** inside that same trace (e.g. `search_manuals`) — showing the exact query the model generated and the manual excerpt it got back.
3. **The root run's Metadata/Tags panel** — showing `machine_id: VFD-06`, `tools_available`, and `llm_model: gpt-4.1-mini`, proving the trace carries real request context, not just raw text.

Don't screenshot or commit the raw `LANGSMITH_API_KEY` — this notebook only ever prints `bool(...)` of it, on purpose, since notebook outputs get committed to the repo. The run URLs themselves embed your org/tenant UUID, which is harmless to share but worth knowing is in there.

## Milestone checkpoint

Core requirement 3 (Tracing/observability) is closed:
- `factory_floor/tracing.py` — `configure_tracing()` (dedicated `factory-floor` LangSmith project, without touching the shared `.env`), `trace_config()` (consistent run naming/tags/metadata), `run_url()` (a single helper that builds a permalink from a `run_id` for *both* `run_diagnostic_agent()` and `ask()`, sidestepping `LangChainTracer.get_run_url()`'s `"No traced run found"` failure on `@traceable`-decorated calls and `Client.read_run()`'s deprecation/`project_id`+`start_time` requirement).
- `factory_floor/agent.py::run_diagnostic_agent()` and `factory_floor/rag.py::ask()` both tag, name, and return a `run_id` for every call.
- Verified live: a real multi-step trace with 2 different tool calls, a baseline trace with none, and a zero-tool-call reasoning trace — three distinct, real shapes in the same project.

Not done here, deliberately out of scope for this notebook:
- **Safety Validator** and the **evaluation notebook** — the other 2 remaining gaps, covered in `notebooks/09_safety_validator.ipynb` and `notebooks/10_evaluation_baseline.ipynb`.
- Taking the actual screenshots for the presentation deliverable — see the list above; that's a manual step tied to your own LangSmith login, not something this notebook can do for you.